# 💊 HỆ THỐNG NHẬN DIỆN THUỐC VÀ ĐỐI SOÁT TƯƠNG TÁC DƯỢC LÝ
### Multiple Pill Recognition & Clinical Drug-Drug Interaction Safety Platform
---
Notebook này tự động thiết lập và chạy toàn bộ hệ thống gồm:
1. **Mô hình AI Computer Vision**: YOLOv11 Segmentation + ResNet-18 Attribute Multi-Head + PaddleOCR
2. **Cơ sở dữ liệu Dược thư Quốc gia (RxNorm CSDL)**: Định danh thuốc, mã khắc, hoạt chất, ma trận DDI
3. **Giao diện Lâm sàng Web & Mobile**: Streamlit + Cloudflare Tunnel (Truy cập trực tiếp không cần mật khẩu)
---
## 🔹 BƯỚC 1: Clone Mã Nguồn (Nhánh mergerBe/Fe) & Cài Đặt Dependencies

In [ ]:
import os
import shutil

# 1. Luôn quay về thư mục an toàn trước khi xóa repo cũ
%cd /kaggle/working
if os.path.exists('/kaggle/working/repo'):
    shutil.rmtree('/kaggle/working/repo')

# 2. Clone nhánh mergerBe/Fe chuẩn
!git clone -b mergerBe/Fe https://github.com/GOx9-P/Multiple-Pill-Recognition-And-Interaction-Safety.git /kaggle/working/repo

# 3. Chuyển vào thư mục repo
%cd /kaggle/working/repo

# 4. Cài đặt các thư viện phụ thuộc
!pip install -q --upgrade pip
!pip install -q -r requirements.txt
!pip install -q streamlit ultralytics paddleocr paddlepaddle-gpu kagglehub pyngrok

print("✅ BƯỚC 1 HOÀN TẤT: Đã clone mã nguồn và cài đặt toàn bộ package thành công!")

---
## 🔹 BƯỚC 2: Tự Động Quét & Nạp Model Weights (YOLOv11, ResNet-18, CSDL)

In [ ]:
import os
import shutil
import glob
from pathlib import Path

os.chdir('/kaggle/working/repo')
repo_root = Path('/kaggle/working/repo')
seg_dir = repo_root / 'models/segmentation_yolov11_full_finetune'
attr_dir = repo_root / 'models/attribute_resnet18_last_blocks_finetune'
db_seed_dir = repo_root / 'database_seed'

seg_dir.mkdir(parents=True, exist_ok=True)
attr_dir.mkdir(parents=True, exist_ok=True)
db_seed_dir.mkdir(parents=True, exist_ok=True)

print("🔍 Đang tự động quét & liên kết toàn bộ Model Artifacts...")

# 1. Quét tìm tất cả các file .pt trên toàn bộ /kaggle/input
found_seg = False
found_attr = False

for pt in glob.glob('/kaggle/input/**/*.pt', recursive=True):
    bname = os.path.basename(pt).lower()
    if 'yolo' in bname or 'seg' in bname:
        dest = seg_dir / 'yolov11m_seg_mediseg_full_finetune_v1.pt'
        shutil.copy(pt, dest)
        print(f"  ✓ Đã nạp YOLOv11 Segmentation: {pt} -> {dest}")
        found_seg = True
    elif 'best' in bname or 'attr' in bname or 'resnet' in bname:
        dest = attr_dir / 'best.pt'
        shutil.copy(pt, dest)
        print(f"  ✓ Đã nạp ResNet-18 Attribute: {pt} -> {dest}")
        found_attr = True

# Tự động tải qua kagglehub nếu chưa có trong /kaggle/input
if not found_seg or not found_attr:
    try:
        import kagglehub
        if not found_seg:
            print("  ⏳ Đang tải YOLOv11 qua link kagglehub...")
            p = kagglehub.dataset_download('nnphuchcmus/pill-segmentation-model')
            for f in glob.glob(f'{p}/**/*.pt', recursive=True):
                shutil.copy(f, seg_dir / 'yolov11m_seg_mediseg_full_finetune_v1.pt')
                print("  ✓ Đã nạp YOLOv11 Segmentation từ kagglehub!")
                found_seg = True
                break
        if not found_attr:
            print("  ⏳ Đang tải ResNet-18 qua link kagglehub...")
            p = kagglehub.dataset_download('nnphuchcmus/attrubute-artifact')
            for f in glob.glob(f'{p}/**/*', recursive=True):
                if os.path.isfile(f):
                    shutil.copy(f, attr_dir / os.path.basename(f))
            print("  ✓ Đã nạp ResNet-18 Artifacts từ kagglehub!")
            found_attr = True
    except Exception as e:
        print(f"  ⚠️ Lưu ý tải hub: {e}")

# 2. Quét liên kết cấu hình nhãn, ngưỡng cho Attribute ResNet-18
copied_cfg = 0
for f in glob.glob('/kaggle/input/**/*', recursive=True):
    if os.path.isfile(f):
        bname = os.path.basename(f).lower()
        if 'mapping' in bname and bname.endswith('.json'):
            shutil.copy(f, attr_dir / 'label_mapping.json')
            copied_cfg += 1
        elif 'threshold' in bname and bname.endswith('.json'):
            shutil.copy(f, attr_dir / 'optimal_thresholds.json')
            copied_cfg += 1
        elif 'model_config' in bname or ('config' in bname and bname.endswith('.yaml')):
            shutil.copy(f, attr_dir / 'model_config.yaml')
            copied_cfg += 1
if copied_cfg > 0:
    print(f"  ✓ Đã đồng bộ {copied_cfg} files cấu hình nhãn & ngưỡng!")

# 3. Quét đồng bộ dữ liệu CSDL Dược thư
for f in glob.glob('/kaggle/input/**/*.json', recursive=True):
    if 'database' in f.lower() or 'seed' in f.lower():
        shutil.copy(f, db_seed_dir / os.path.basename(f))

# 4. Ghi file cấu hình môi trường .env
with open(repo_root / '.env', 'w', encoding='utf-8') as f:
    f.write('DATABASE_URL=sqlite:///./medication.db\n')
    f.write('LLM_PROVIDER=fallback\n')

print("\n🎉 BƯỚC 2 HOÀN TẤT: Toàn bộ Models và CSDL đã được nạp và liên kết thành công!")

---
## 🔹 BƯỚC 3: Khởi Tạo & Nạp CSDL Dược Thư (SQLite Database)

In [ ]:
import sys
import os

os.chdir('/kaggle/working/repo')
src_path = '/kaggle/working/repo/src'
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = f"{src_path}:{os.environ.get('PYTHONPATH', '')}"

# Chạy seed database
!PYTHONPATH=src python -m pill_safety.database.scripts.seed

from pill_safety.database.session import SessionLocal
from pill_safety.database.models import DrugProduct, DrugInteraction

with SessionLocal() as db:
    total_drugs = db.query(DrugProduct).count()
    total_ddi = db.query(DrugInteraction).count()
    print("=" * 60)
    print(f"📊 CSDL ĐÃ NẠP THÀNH CÔNG: {total_drugs} sản phẩm thuốc | {total_ddi} cặp tương tác DDI")
    print("=" * 60)

print("✅ BƯỚC 3 HOÀN TẤT 100%!")

---
## 🔹 BƯỚC 4: Khởi Chạy Web & Mobile UI (Cloudflare Tunnel)

In [ ]:
import subprocess
import time
import os

os.chdir('/kaggle/working/repo')

# 1. Tắt các tiến trình Streamlit cũ nếu có
!fuser -k 8501/tcp 2>/dev/null || true

# 2. Tải và cài đặt Cloudflared (nếu chưa có)
!which cloudflared > /dev/null 2>&1 || (wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1)

# 3. Khởi chạy Streamlit ở background
cmd_streamlit = "streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false --server.enableWebsocketCompression false"
subprocess.Popen(cmd_streamlit, shell=True)
time.sleep(3)

print("🚀 Streamlit đã khởi chạy thành công!")
print("🌐 Đang mở đường link Cloudflare Public Tunnel...")

# 4. Mở tunnel Cloudflare trực tiếp (Click link https://*.trycloudflare.com bên dưới để mở web)
!cloudflared tunnel --url http://localhost:8501